# 00. A2A 协议：Agent 与 Agent 如何对话

MCP 让 agent 接工具，但两个独立 agent（可能不同框架、不同厂商）如何协作？A2A 的答案：
- **Agent Card**：JSON 名片，发布在 `/.well-known/agent.json`，声明能力（skills）与端点
- **Task**：协作的基本单元，生命周期 submitted → working → completed/failed
- **JSON-RPC 2.0**：`tasks/send`、`tasks/get` 等方法的消息信封

关键设计：A2A 中的 agent 是 **opaque** 的 — 只交换 message 和 artifact，不暴露内部 prompt/记忆。

> **检查点**：能解释为什么 A2A 选择 Task 而非直接 chat completion 作为交互单元（长任务、异步、多轮追加）。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()

🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent

SCRIPT = ROOT / 'agents' / 'protocols' / 'scripts' / 'a2a_pair_demo.py'
result = subprocess.run([sys.executable, str(SCRIPT), '--task', 'research', '--verbose'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)

=== Agent Card discovery ===
GET a2a://local/researcher/.well-known/agent.json
{
  "name": "researcher",
  "url": "a2a://local/researcher",
  "description": "Finds and verifies facts on a topic.",
  "skills": [
    {
      "id": "research",
      "name": "Research",
      "description": "Gather key facts about a topic"
    }
  ],
  "capabilities": {
    "streaming": false,
    "pushNotifications": false
  }
}

GET a2a://local/writer/.well-known/agent.json
{
  "name": "writer",
  "url": "a2a://local/writer",
  "description": "Turns raw material into polished prose.",
  "skills": [
    {
      "id": "summarize",
      "name": "Summarize",
      "description": "Write a short summary from findings"
    }
  ],
  "capabilities": {
    "streaming": false,
    "pushNotifications": false
  }
}

=== tasks/send → researcher ===
{
  "jsonrpc": "2.0",
  "id": "c527c271",
  "method": "tasks/send",
  "params": {
    "id": "61aa5fef",
    "message": {
      "role": "user",
      "parts": [
        {
 

## A2A vs MCP 的边界

| 维度 | MCP | A2A |
|------|-----|-----|
| 交互双方 | agent ↔ tool/resource | agent ↔ agent |
| 对端模型 | 无智能（工具按 schema 执行） | 有智能（opaque agent 自主完成 task） |
| 消息格式 | JSON-RPC（tools/list, tools/call） | JSON-RPC（tasks/send, tasks/get） |
| 发现机制 | 客户端配置 server | Agent Card（.well-known） |
| 典型延迟 | 毫秒级 | 秒到分钟级（agent 要思考） |

反模式：把另一个 agent 包装成 MCP tool — 丢失 Task 生命周期与异步能力。

In [3]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent

SCRIPT = ROOT / 'agents' / 'protocols' / 'scripts' / 'a2a_pair_demo.py'
result = subprocess.run([sys.executable, str(SCRIPT), '--task', 'research', '--use-api', '--verbose'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)

=== Agent Card discovery ===
GET a2a://local/researcher/.well-known/agent.json
{
  "name": "researcher",
  "url": "a2a://local/researcher",
  "description": "Finds and verifies facts on a topic.",
  "skills": [
    {
      "id": "research",
      "name": "Research",
      "description": "Gather key facts about a topic"
    }
  ],
  "capabilities": {
    "streaming": false,
    "pushNotifications": false
  }
}

GET a2a://local/writer/.well-known/agent.json
{
  "name": "writer",
  "url": "a2a://local/writer",
  "description": "Turns raw material into polished prose.",
  "skills": [
    {
      "id": "summarize",
      "name": "Summarize",
      "description": "Write a short summary from findings"
    }
  ],
  "capabilities": {
    "streaming": false,
    "pushNotifications": false
  }
}

=== tasks/send → researcher ===
{
  "jsonrpc": "2.0",
  "id": "d05b7ef7",
  "method": "tasks/send",
  "params": {
    "id": "70ebebbd",
    "message": {
      "role": "user",
      "parts": [
        {
 

## 真实双 Agent 观察

API 模式下 researcher 和 writer 都是真实 LLM：注意 researcher 的 findings 作为 artifact
原样流入 writer 的 task — 这就是「artifact 传递」模式，agent 之间不需要共享 context window。

## 练习

1. 给 `WriterAgent` 增加一个 skill `translate`，让 orchestrator 把 findings 先翻译再总结（三个 agent 串联）。
2. A2A 支持一个 task 内多轮 message（client 追问）。修改 demo：writer 完成后，client 在同一 task 里追加「再短一点」。
3. 思考：什么时候该用 A2A 而不是在一个进程里写两个函数？（提示：团队边界、框架边界、信任边界）